In [1]:
import time
import json
import csv

from qiskit import QuantumCircuit, QuantumRegister
from qiskit.transpiler import CouplingMap, PassManager
from pathlib import Path
from typing import Dict, List, Iterable
from util import EAGLE_COUPLING, sabre, count_swaps 
from multilevel_sabre import MultiLevelSabre
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.transpiler.passes import SabreLayout, SetLayout, SabreSwap, RemoveFinalMeasurements
from qiskit.converters import circuit_to_dag, dag_to_circuit

random_seed = 1

Coupling map

In [2]:
from util import EAGLE_COUPLING

coupling_map = CouplingMap(couplinglist=EAGLE_COUPLING)
coupling_map.make_symmetric()

Select Circuit

In [3]:
filename = "circuits/dnn_n51.qasm"
qc = QuantumCircuit.from_qasm_file(filename)
qc_dag = circuit_to_dag(qc)
qc_dag = RemoveFinalMeasurements().run(qc_dag)
qc = dag_to_circuit(qc_dag)

For layout considerations

In [4]:
num_program_qubit=qc.num_qubits
num_classical_bits=qc.num_clbits
num_physical_qubit=max(max(i) for i in coupling_map)+1
if num_physical_qubit>num_program_qubit:
    temp_qc=QuantumCircuit(num_physical_qubit,num_classical_bits)
    temp_qc.compose(qc,inplace=True)
    qc=temp_qc

In [5]:
qr = qc.qregs[0]
qr

QuantumRegister(127, 'q')

Sabre PM 

In [6]:
pm_sabre = generate_preset_pass_manager(optimization_level=3, coupling_map=coupling_map, seed_transpiler=random_seed)
qc_tr_sabre = pm_sabre.run(qc)

swaps_sabre = count_swaps(qc_tr_sabre)
depth_2q_sabre = qc_tr_sabre.depth(lambda x: x.operation.num_qubits == 2)
print(f"SABRE: SWAPs={swaps_sabre}, 2Q Depth={depth_2q_sabre}")


SABRE: SWAPs=6, 2Q Depth=65


SABRE ML PM

In [7]:
pm_ml_sabre = PassManager([
    MultiLevelSabre(
        coupling_graph=coupling_map,
        cycles=10,
        random_seed=1,
        coarsest_solving_trials=50,
        num_interpolation=10,
        use_initial_embedding=True,
        verbose=0
    )
])
qc_tr_ml_sabre = pm_ml_sabre.run(qc)

swaps_ml_sabre = count_swaps(qc_tr_ml_sabre)
depth_2q_ml_sabre = qc_tr_ml_sabre.depth(lambda x: x.operation.num_qubits == 2)


In [8]:
print(f"ML-SABRE: SWAPs={swaps_ml_sabre}, 2Q Depth={depth_2q_ml_sabre}")

ML-SABRE: SWAPs=0, 2Q Depth=52


In [13]:
qc_tr_ml_sabre.count_ops()

OrderedDict([('ry', 50),
             ('rz', 50),
             ('cswap', 25),
             ('rzz', 24),
             ('cry', 24),
             ('crz', 24),
             ('h', 2),
             ('ryy', 1),
             ('ryy_140578989711904', 1),
             ('ryy_140578989712096', 1),
             ('ryy_140578989712192', 1),
             ('ryy_140578989712288', 1),
             ('ryy_140578989712384', 1),
             ('ryy_140578989712480', 1),
             ('ryy_140578989712576', 1),
             ('ryy_140578989712672', 1),
             ('ryy_140578989712768', 1),
             ('ryy_140578989712864', 1),
             ('ryy_140578989712960', 1),
             ('ryy_140578989713056', 1),
             ('ryy_140578989713152', 1),
             ('ryy_140578989713248', 1),
             ('ryy_140578989713344', 1),
             ('ryy_140578989713440', 1),
             ('ryy_140578989713536', 1),
             ('ryy_140578989713632', 1),
             ('ryy_140578989713728', 1),
             ('ry

In [9]:
ml_layout = {0: 15, 1: 57, 2: 56, 3: 52, 4: 37, 5: 38, 6: 39, 7: 33, 8: 20, 9: 19, 10: 18, 11: 14, 12: 0, 13: 1, 14: 2, 15: 3, 16: 4, 17: 5, 18: 6, 19: 7, 20: 8, 21: 9, 22: 10, 23: 11, 24: 12, 25: 13, 26: 16, 27: 17, 28: 21, 29: 22, 30: 23, 31: 24, 32: 25, 33: 26, 34: 27, 35: 28, 36: 29, 37: 30, 38: 31, 39: 32, 40: 34, 41: 35, 42: 36, 43: 40, 44: 41, 45: 42, 46: 43, 47: 44, 48: 45, 49: 46, 50: 47, 51: 48, 52: 49, 53: 50, 54: 51, 55: 53, 56: 54, 57: 55, 58: 58, 59: 59, 60: 60, 61: 61, 62: 62, 63: 63, 64: 64, 65: 65, 66: 66, 67: 67, 68: 68, 69: 69, 70: 70, 71: 71, 72: 72, 73: 73, 74: 74, 75: 75, 76: 76, 77: 77, 78: 78, 79: 79, 80: 80, 81: 81, 82: 82, 83: 83, 84: 84, 85: 85, 86: 86, 87: 87, 88: 88, 89: 89, 90: 90, 91: 91, 92: 92, 93: 93, 94: 94, 95: 95, 96: 96, 97: 97, 98: 98, 99: 99, 100: 100, 101: 101, 102: 102, 103: 103, 104: 104, 105: 105, 106: 106, 107: 107, 108: 108, 109: 109, 110: 110, 111: 111, 112: 112, 113: 113, 114: 114, 115: 115, 116: 116, 117: 117, 118: 118, 119: 119, 120: 120, 121: 121, 122: 122, 123: 123, 124: 124, 125: 125, 126: 126}

In [10]:
from qiskit.transpiler import Layout


layout = Layout({ qr[v]: p for v, p in ml_layout.items() })

Sabre PM with ML layout

In [11]:
pm_sabre_ml_layout = generate_preset_pass_manager(optimization_level=3, coupling_map=coupling_map, seed_transpiler=random_seed,
                                                initial_layout=layout)

In [12]:
qc_tr_sabre_ml_layout = pm_sabre_ml_layout.run(qc)
swaps_sabre_ml_layout = count_swaps(qc_tr_sabre_ml_layout)
depth_2q_sabre_ml_layout = qc_tr_sabre_ml_layout.depth(lambda x: x.operation.num_qubits == 2)

print(f"SABRE with ML layout: SWAPs={swaps_sabre_ml_layout}, 2Q Depth={depth_2q_sabre_ml_layout}")


SABRE with ML layout: SWAPs=0, 2Q Depth=52


In [14]:
qc_tr_sabre_ml_layout.count_ops()

OrderedDict([('u3', 50),
             ('cswap', 25),
             ('rzz', 24),
             ('cry', 24),
             ('crz', 24),
             ('h', 2),
             ('ryy', 1),
             ('ryy_140578989711904', 1),
             ('ryy_140578989712096', 1),
             ('ryy_140578989712192', 1),
             ('ryy_140578989712288', 1),
             ('ryy_140578989712384', 1),
             ('ryy_140578989712480', 1),
             ('ryy_140578989712576', 1),
             ('ryy_140578989712672', 1),
             ('ryy_140578989712768', 1),
             ('ryy_140578989712864', 1),
             ('ryy_140578989712960', 1),
             ('ryy_140578989713056', 1),
             ('ryy_140578989713152', 1),
             ('ryy_140578989713248', 1),
             ('ryy_140578989713344', 1),
             ('ryy_140578989713440', 1),
             ('ryy_140578989713536', 1),
             ('ryy_140578989713632', 1),
             ('ryy_140578989713728', 1),
             ('ryy_140578989713824', 1),
 